In [1]:
import pandas as pd
import numpy as np
import os, re
import itertools
from IPython.display import display

In [2]:
import matplotlib.pyplot as plt
import seaborn as sns

In [5]:
sample_dir = '../pkgs/samples'
data_path = '../results/npm-exp/run_table.csv'

In [6]:
data = pd.read_csv(data_path)
data['energy-total'] = data['energy-pkg'] + data['energy-ram'] 
data.head()

,__run_id,__done,alg,subject,energy-pkg,energy-ram,execution-time,energy-total
0,run_206_repetition_11,DONE,zstd,zstd/babel-plugin-check-es2015-constants--babe...,0.26,0.03,0.003511,0.29
1,run_73_repetition_12,DONE,xz,xz/@sindresorhus__is--is-8.1.0.tar.xz,0.41,0.05,0.005875,0.46
2,run_251_repetition_20,DONE,zstd,zstd/xml2js--xml2js-0.6.2.tar.zst,0.97,0.11,0.014525,1.08
3,run_220_repetition_11,DONE,zstd,zstd/iconv-lite--iconv-lite-0.7.2.tar.zst,0.36,0.04,0.005436,0.40
4,run_143_repetition_13,DONE,zopfli,zopfli/babel-plugin-check-es2015-constants--ba...,0.26,0.03,0.004009,0.29


In [7]:
labels = data['alg'].unique()

In [11]:
samples = {k : pd.read_csv(f'{sample_dir}/{k}.csv') for k in labels}
samp

# Data Preprocessing

In [12]:
def clean_name(path: str) -> str:
    # Remove everything up to the last "/"
    base = os.path.basename(path)
    base = re.sub(r'\.tar(\.[a-zA-Z0-9]+)?$|\.tgz$', '', base)
    return base

## Split run number and ID 

In [13]:
# Preprocessing to split run and rep id; Step not needed in the next version of Experiment Runner 
import re

run_ids = data['__run_id'].to_list()
def extract_run_reps(run_ids):
    run_reps = []
 
    for rid in run_ids:
        run, reps = re.findall(r'\d+', rid)
        run_reps.append({'__run_id' : rid, 'run' : run, 'reps' : reps})
 
    return run_reps

run_reps = extract_run_reps(run_ids)
df_run_reps = pd.DataFrame(run_reps)
data = pd.merge(data, df_run_reps, on=['__run_id'])
data.head()

,__run_id,__done,alg,subject,energy-pkg,energy-ram,execution-time,energy-total,run,reps
0,run_206_repetition_11,DONE,zstd,zstd/babel-plugin-check-es2015-constants--babe...,0.26,0.03,0.003511,0.29,206,11
1,run_73_repetition_12,DONE,xz,xz/@sindresorhus__is--is-8.1.0.tar.xz,0.41,0.05,0.005875,0.46,73,12
2,run_251_repetition_20,DONE,zstd,zstd/xml2js--xml2js-0.6.2.tar.zst,0.97,0.11,0.014525,1.08,251,20
3,run_220_repetition_11,DONE,zstd,zstd/iconv-lite--iconv-lite-0.7.2.tar.zst,0.36,0.04,0.005436,0.40,220,11
4,run_143_repetition_13,DONE,zopfli,zopfli/babel-plugin-check-es2015-constants--ba...,0.26,0.03,0.004009,0.29,143,13


## Add size
size is the same across repetitions thus size is associate to each run 

# Boxplot aggregated data

In [ ]:
#data = data[data['size'] < 5]
groups = data.groupby(['alg', 'run'])['energy-total'].describe()
groups['cv'] = groups['std'] / groups['mean']
groups

In [ ]:
values = [
    groups.xs(k, level='alg')['50%'].tolist() for k in labels 
]

In [ ]:
plt.figure(figsize=(10, 5.7), dpi=300)

positions = range(1, len(labels) + 1)

vp = plt.violinplot(values, positions=positions, widths=0.8,
                     showmeans=False, showextrema=False, showmedians=False)
bp = plt.boxplot(values, positions=positions, widths=0.3,
                  patch_artist=True)

plt.setp(bp['fliers'], marker='o', markersize=4,
          markerfacecolor='none',  
          markeredgecolor='black',
          markeredgewidth=0.8,
          alpha=0.6)

# Customize boxplots
for element in ['whiskers', 'fliers', 'means', 'medians', 'caps']:
    plt.setp(bp[element], color='black')
for box in bp['boxes']:
    box.set(facecolor='white', edgecolor='black')

# overlay all raw points with jitter
rng = np.random.default_rng(0)
for pos, y in zip(positions, values):
    x = rng.normal(pos, 0.05, size=len(y))  # small horizontal jitter
    plt.scatter(x, y, s=10, facecolors='none', edgecolors='black',
                linewidths=0.6, alpha=0.5, zorder=3)

plt.xticks(positions, labels)

plt.tight_layout()
plt.show()

# Calculate Energy/Time per Byte